# Data Source Ingestion & Authority Reconciliation
Loads folders, ground truth, and metadata, merging them to spot conflicts.
Paths come from 01_config.

In [1]:
# Pull all variables from the config notebook
%run 01_config.ipynb

import os
import pandas as pd
from pathlib import Path

Creating output structure in: C:\SKIN CANCER v2\pipe output
  11 subdirectories ready.
Set numpy/random seeds to 42.
Optional torch seeds set.
  01_config -- FINAL SUMMARY

Frozen paths:
  DATASET_ROOT       : C:\SKIN CANCER v2\DS
  NV_DIR             : C:\SKIN CANCER v2\DS\NV
  MEL_DIR            : C:\SKIN CANCER v2\DS\MEL
  BCC_DIR            : C:\SKIN CANCER v2\DS\BCC
  GROUND_TRUTH_CSV   : C:\SKIN CANCER v2\DS\ISIC_2019_Training_GroundTruth.csv
  METADATA_CSV       : C:\SKIN CANCER v2\DS\ISIC_2019_Training_Metadata.csv
  OUTPUT_ROOT        : C:\SKIN CANCER v2\pipe output
  FINAL_DATASET_ROOT : C:\SKIN CANCER v2\final DS

Class mapping      : {'NV': 0, 'MEL': 1, 'BCC': 2}

Output folders created:
  [OK] manifests
  [OK] reconciliation_reports
  [OK] duplicate_review
  [OK] audit_reports
  [OK] visual_inspection
  [OK] splits
  [OK] training_logs
  [OK] evaluation
  [OK] explainability
  [OK] robustness
  [OK] models

TensorFlow removed          : True
torch optional (not req 01-07):

## Base ID Suffix Extraction

In [2]:
def get_base_id_and_suffix(file_stem_raw):
    for suffix in APPROVED_SUFFIXES:
        if file_stem_raw.endswith(suffix):
            return file_stem_raw[:-len(suffix)], suffix
    return file_stem_raw, ""


## Folder Scanning

In [3]:
def scan_folders():
    records = []
    folders_to_scan = [(NV_DIR, "NV"), (MEL_DIR, "MEL"), (BCC_DIR, "BCC")]
    ext_counts = {}
    for folder_path, raw_folder_label in folders_to_scan:
        if not os.path.isdir(folder_path):
            print(f"  WARNING: folder not found: {folder_path}")
            continue
        for file in os.listdir(folder_path):
            ext = Path(file).suffix.lower()
            ext_counts[ext] = ext_counts.get(ext, 0) + 1
            if ext not in ALLOWED_EXTENSIONS:
                continue
            full_path = os.path.join(folder_path, file)
            file_stem_raw = Path(file).stem
            file_exists = os.path.exists(full_path)
            file_size_bytes = os.path.getsize(full_path) if file_exists else 0
            file_accessible_status = os.access(full_path, os.R_OK) if file_exists else False
            base_id_candidate, recognized_suffix = get_base_id_and_suffix(file_stem_raw)
            records.append({
                "full_path": full_path,
                "folder_name": os.path.basename(folder_path),
                "raw_folder_label": raw_folder_label,
                "file_name_with_extension": file,
                "file_stem_raw": file_stem_raw,
                "base_id_candidate": base_id_candidate,
                "recognized_suffix": recognized_suffix,
                "extension": ext,
                "file_exists": file_exists,
                "file_size_bytes": file_size_bytes,
                "file_accessible_status": file_accessible_status
            })
    df_files = pd.DataFrame(records)
    print(f"Scanned {len(df_files):,} valid image files.")
    print("Extension counts (all files including unsupported):")
    for ext, cnt in sorted(ext_counts.items(), key=lambda x: -x[1]):
        tag = "allowed" if ext in ALLOWED_EXTENSIONS else "unsupported"
        print(f"  {ext:<10}: {cnt:,}  ({tag})")
    return df_files, ext_counts

df_files, ext_counts = scan_folders()
df_files.head(3)

Scanned 20,720 valid image files.
Extension counts (all files including unsupported):
  .jpg      : 20,720  (allowed)


,full_path,folder_name,raw_folder_label,file_name_with_extension,file_stem_raw,base_id_candidate,recognized_suffix,extension,file_exists,file_size_bytes,file_accessible_status
0,C:\SKIN CANCER v2\DS\NV\ISIC_0000000.jpg,NV,NV,ISIC_0000000.jpg,ISIC_0000000,ISIC_0000000,,.jpg,True,49964,True
1,C:\SKIN CANCER v2\DS\NV\ISIC_0000001.jpg,NV,NV,ISIC_0000001.jpg,ISIC_0000001,ISIC_0000001,,.jpg,True,38941,True
2,C:\SKIN CANCER v2\DS\NV\ISIC_0000003.jpg,NV,NV,ISIC_0000003.jpg,ISIC_0000003,ISIC_0000003,,.jpg,True,45774,True


## Ground Truth Extraction

In [4]:
def load_ground_truth():
    df_gt = pd.read_csv(GROUND_TRUTH_CSV)
    gt_records = []
    invalid_records = []
    for _, row in df_gt.iterrows():
        csv_image_id = row["image"]
        active_classes = [c for c in CLASS_NAMES if c in row and row[c] == 1.0]
        if len(active_classes) == 1:
            gt_label = active_classes[0]
            validity = "valid"
        elif len(active_classes) == 0:
            gt_label = "none"
            validity = "no_active_label"
        else:
            gt_label = "multiple"
            validity = "multiple_active_labels"
        unsupported = [c for c in row.index if c not in CLASS_NAMES and c != "image"]
        for unsupp_c in unsupported:
            if row[unsupp_c] == 1.0 and validity == "valid":
                validity = "unsupported_label"
        if validity == "valid":
            gt_records.append({"csv_image_id": csv_image_id, "gt_label": gt_label, "gt_row_validity": validity})
        else:
            rec = {"image": csv_image_id, "invalid_reason": validity}
            for c in row.index:
                if c != "image": rec[c] = row[c]
            invalid_records.append(rec)
    df_gt_clean = pd.DataFrame(gt_records)
    print(f"Loaded {len(df_gt_clean):,} valid ground truth rows (NV, MEL, BCC).")
    if len(invalid_records) > 0:
        df_invalid = pd.DataFrame(invalid_records)
        inv_path = os.path.join(OUTPUT_ROOT, "manifests", "ground_truth_invalid_rows.csv")
        df_invalid.to_csv(inv_path, index=False)
        print(f"Exported {len(df_invalid):,} invalid ground truth rows to {inv_path}")
    return df_gt_clean

df_gt = load_ground_truth()
df_gt.head(3)

Loaded 20,720 valid ground truth rows (NV, MEL, BCC).
Exported 4,611 invalid ground truth rows to C:\SKIN CANCER v2\pipe output\manifests\ground_truth_invalid_rows.csv


,csv_image_id,gt_label,gt_row_validity
0,ISIC_0000000,NV,valid
1,ISIC_0000001,NV,valid
2,ISIC_0000002,MEL,valid


## Metadata Extraction

In [5]:
def load_metadata():
    df_meta = pd.read_csv(METADATA_CSV)
    cols = ["image", "age_approx", "sex", "anatom_site_general", "lesion_id"]
    for c in cols:
        if c not in df_meta.columns:
            df_meta[c] = None
    df_meta = df_meta[cols]
    df_meta.rename(columns={"image": "csv_image_id"}, inplace=True)
    df_meta["metadata_row_found"] = True
    return df_meta

df_meta = load_metadata()
df_meta.head(3)

,csv_image_id,age_approx,sex,anatom_site_general,lesion_id,metadata_row_found
0,ISIC_0000000,55.0,female,anterior torso,NaN,True
1,ISIC_0000001,30.0,female,anterior torso,NaN,True
2,ISIC_0000002,60.0,female,upper extremity,NaN,True


## Reconciliation & Report Saving

In [6]:
def reconcile_sources(df_files, df_gt, df_meta):
    merged_records = []
    gt_dict = df_gt.set_index("csv_image_id")["gt_label"].to_dict()
    for _, file_row in df_files.iterrows():
        stem, base_id = file_row["file_stem_raw"], file_row["base_id_candidate"]
        folder_label  = file_row["raw_folder_label"]
        match_status, matched_csv_image_id, gt_label = "unmatched", None, None
        if stem in gt_dict:
            match_status, matched_csv_image_id, gt_label = "matched_exact", stem, gt_dict[stem]
        elif base_id in gt_dict:
            match_status, matched_csv_image_id, gt_label = "matched_via_suffix_rule", base_id, gt_dict[base_id]
        if gt_label is None:
            label_agreement_status = "missing_ground_truth"
        elif gt_label == folder_label:
            label_agreement_status = "agree"
        else:
            label_agreement_status = "disagree"
        rec = file_row.to_dict()
        rec.update({
            "match_status": match_status,
            "matched_csv_image_id": matched_csv_image_id,
            "gt_label": gt_label,
            "folder_label": folder_label,
            "label_agreement_status": label_agreement_status
        })
        merged_records.append(rec)
    df_reconciled = pd.DataFrame(merged_records)
    df_final = pd.merge(
        df_reconciled, df_meta,
        left_on="matched_csv_image_id", right_on="csv_image_id", how="left"
    )
    df_final["metadata_row_found"] = df_final["metadata_row_found"].fillna(False)
    if "csv_image_id" in df_final.columns:
        df_final.drop(columns=["csv_image_id"], inplace=True)

    def determine_review_status(row):
        if row["match_status"] == "unmatched" or row["label_agreement_status"] == "disagree":
            return "manual_review_required"
        if row["label_agreement_status"] == "agree":
            return "provisionally_approved"
        return "manual_review_required"

    df_final["review_status"]   = df_final.apply(determine_review_status, axis=1)
    df_final["review_decision"] = None
    df_final["review_notes"]    = None

    manifest_path = os.path.join(OUTPUT_ROOT, "manifests", "authoritative_raw_manifest.csv")
    df_final.to_csv(manifest_path, index=False)
    print(f"Saved authoritative raw manifest to {manifest_path}")

    df_needs_review = df_final[df_final["review_status"] == "manual_review_required"]
    review_path = os.path.join(OUTPUT_ROOT, "reconciliation_reports", "reconciliation_review.csv")
    review_cols = [
        "full_path", "file_name_with_extension", "file_stem_raw", "base_id_candidate",
        "folder_label", "matched_csv_image_id", "gt_label", "match_status",
        "label_agreement_status", "age_approx", "sex", "anatom_site_general",
        "lesion_id", "review_status", "review_decision", "review_notes"
    ]
    df_needs_review[[c for c in review_cols if c in df_needs_review.columns]].to_csv(review_path, index=False)
    print(f"Generated reconciliation review report: {len(df_needs_review):,} items.")
    return df_final

df_reconciled = reconcile_sources(df_files, df_gt, df_meta)
df_reconciled.head(3)

Saved authoritative raw manifest to C:\SKIN CANCER v2\pipe output\manifests\authoritative_raw_manifest.csv
Generated reconciliation review report: 0 items.


,full_path,folder_name,raw_folder_label,file_name_with_extension,file_stem_raw,base_id_candidate,recognized_suffix,extension,file_exists,file_size_bytes,...,folder_label,label_agreement_status,age_approx,sex,anatom_site_general,lesion_id,metadata_row_found,review_status,review_decision,review_notes
0,C:\SKIN CANCER v2\DS\NV\ISIC_0000000.jpg,NV,NV,ISIC_0000000.jpg,ISIC_0000000,ISIC_0000000,,.jpg,True,49964,...,NV,agree,55.0,female,anterior torso,NaN,True,provisionally_approved,None,None
1,C:\SKIN CANCER v2\DS\NV\ISIC_0000001.jpg,NV,NV,ISIC_0000001.jpg,ISIC_0000001,ISIC_0000001,,.jpg,True,38941,...,NV,agree,30.0,female,anterior torso,NaN,True,provisionally_approved,None,None
2,C:\SKIN CANCER v2\DS\NV\ISIC_0000003.jpg,NV,NV,ISIC_0000003.jpg,ISIC_0000003,ISIC_0000003,,.jpg,True,45774,...,NV,agree,30.0,male,upper extremity,NaN,True,provisionally_approved,None,None


In [7]:
# ── Final summary ─────────────────────────────────────────────────────────────
_manifest_path = os.path.join(OUTPUT_ROOT, "manifests", "authoritative_raw_manifest.csv")
_review_path   = os.path.join(OUTPUT_ROOT, "reconciliation_reports", "reconciliation_review.csv")

print("=" * 60)
print("  02_ingestion -- FINAL SUMMARY")
print("=" * 60)
print(f"\nRaw scanned images total: {len(df_files):,}")
for label in CLASS_NAMES:
    cnt = int((df_files["raw_folder_label"] == label).sum())
    print(f"  {label}: {cnt:,}")
print(f"\nAllowed extensions:")
for ext in ALLOWED_EXTENSIONS:
    print(f"  {ext}")
print(f"\nGround truth valid rows : {len(df_gt):,}")
print(f"Metadata matched rows   : {int(df_reconciled['metadata_row_found'].sum()):,}")
print(f"\nReconciliation statuses:")
for status, cnt in df_reconciled["review_status"].value_counts().items():
    print(f"  {status}: {cnt:,}")
print(f"\nOutput file verification:")
for p in [_manifest_path, _review_path]:
    exists = os.path.exists(p)
    size   = os.path.getsize(p) if exists else 0
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {os.path.basename(p):<45} {size:>10,} bytes")
print("=" * 60)

  02_ingestion -- FINAL SUMMARY

Raw scanned images total: 20,720
  NV: 12,875
  MEL: 4,522
  BCC: 3,323

Allowed extensions:
  .jpg
  .jpeg
  .png

Ground truth valid rows : 20,720
Metadata matched rows   : 20,720

Reconciliation statuses:
  provisionally_approved: 20,720

Output file verification:
  [OK] authoritative_raw_manifest.csv                 4,657,009 bytes
  [OK] reconciliation_review.csv                            235 bytes
